# wandb-init-run — worked example 2: Open a wandb run passing a dict as config

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-init-run`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Instead of passing the args dataclass directly to `config=`, you can convert it to a plain dict first with `dataclasses.asdict()`. This is useful when you want to modify the config dict before logging (e.g., adding computed fields or removing private fields). The wandb dashboard treats both forms identically.

## Worked solution

**Step 1 — convert dataclass to dict.**
We use `dataclasses.asdict(args)` to produce a plain Python dict of all fields. This is the same snapshot wandb would create internally from a dataclass, but now we can add or modify entries before passing it.

**Step 2 — augment the config.**
We add a computed entry `'n_params'` to the config dict. This will appear in the wandb config panel alongside the hyperparameters, even though it wasn't on the original dataclass.

**Step 3 — call wandb.init with the dict.**
We pass the augmented dict as `config=`. The wandb run records all entries in the dict. The `project` and `name` still come from the original args object.

In [ ]:
import sys
from unittest.mock import MagicMock
from dataclasses import dataclass, asdict
sys.modules.setdefault('wandb', MagicMock())
import wandb

@dataclass
class Args:
    lr: float = 1e-3
    hidden_size: int = 128
    wandb_project: str = 'mlp-demo'
    wandb_name: str = 'wide-hidden'

def open_run_with_dict_config(args, n_params):
    """Open wandb run, passing an augmented dict as config."""
    config = asdict(args)                  # convert dataclass to plain dict
    config['n_params'] = n_params         # add a computed field
    return wandb.init(
        project=args.wandb_project,
        name=args.wandb_name,
        config=config,
    )

# Exercise it
wandb.init.reset_mock()
args = Args(hidden_size=512)
run = open_run_with_dict_config(args, n_params=100_000)
call_kwargs = wandb.init.call_args.kwargs
print('Config logged:', call_kwargs['config'])
print('n_params in config:', 'n_params' in call_kwargs['config'])
print('hidden_size in config:', call_kwargs['config']['hidden_size'])